# 6.0 Data monitor (run in SageMaker Studio)

Data-quality baseline (statistics + constraints) from the **train** features with `DefaultModelMonitor.suggest_baseline`, then a one-off Model Monitor analyzer job on the **production** features. Batch only: no endpoint, no monitoring schedule. Two billable `ml.m5.large` x1 processing jobs. Outputs: `monitoring/data-quality/baseline/` and `monitoring/data-quality/runs/<ts>/`.

In [ ]:
import sys, pathlib
ROOT = pathlib.Path.cwd()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
import io, json, time
import boto3, numpy as np, pandas as pd
from botocore.exceptions import ClientError
from src.pamap2.config import load_config, s3_uri
from src.pamap2.schema import ACTIVITY_NAMES
from src.pamap2 import evaluation as E
from src.pamap2 import monitoring as M
cfg = load_config()
session = boto3.Session(region_name=cfg["region"])
print("Caller:", session.client("sts").get_caller_identity()["Arn"])
s3 = session.client("s3")
cw = session.client("cloudwatch")
sm = session.client("sagemaker")
SPLITS = s3_uri(cfg, "splits")
REPORTS = s3_uri(cfg, "models", "reports/")
MON = lambda key: s3_uri(cfg, "monitoring", key)
NS = cfg["monitoring_namespace"]

def split_key(uri):
    b, k = uri[len("s3://"):].split("/", 1)
    return b, k
def read_bytes(uri):
    b, k = split_key(uri)
    return s3.get_object(Bucket=b, Key=k)["Body"].read()
def read_bytes_or_none(uri):
    try:
        return read_bytes(uri)
    except ClientError as e:
        if e.response["Error"]["Code"] in ("NoSuchKey", "404"):
            return None
        raise
def read_parquet(name):
    return pd.read_parquet(io.BytesIO(read_bytes(SPLITS + name)))
def read_json(uri):
    return json.loads(read_bytes(uri))
def write_json(uri, obj):
    b, k = split_key(uri)
    s3.put_object(Bucket=b, Key=k, Body=json.dumps(obj, indent=2).encode())
    print("Wrote", uri)
def put_text(uri, text):
    b, k = split_key(uri)
    s3.put_object(Bucket=b, Key=k, Body=text.encode())
    return uri
def exists(uri):
    b, k = split_key(uri)
    try:
        s3.head_object(Bucket=b, Key=k)
        return True
    except ClientError:
        return False
def latest_subprefix(uri_prefix):
    """Newest child 'folder' (timestamped run folders sort lexicographically)."""
    b, k = split_key(uri_prefix)
    r = s3.list_objects_v2(Bucket=b, Prefix=k, Delimiter="/")
    subs = sorted(p["Prefix"] for p in r.get("CommonPrefixes", []))
    return f"s3://{b}/{subs[-1]}" if subs else None
def publish(metrics):
    """metrics: {name: value} -> custom namespace (put_metric_data)."""
    cw.put_metric_data(Namespace=NS, MetricData=[{"MetricName": n, "Value": float(v), "Unit": "None"} for n, v in metrics.items()])
    print("Published", metrics)

def safe_publish(metrics):
    try:
        publish(metrics)
    except Exception as e:   # never mask the original error
        print("WARNING: publish failed:", e)
def report_failure(job_name):
    try:
        print("FailureReason:", sm.describe_processing_job(ProcessingJobName=job_name).get("FailureReason"))
    except Exception as e:
        print("Could not describe job:", e)

FEATURES = read_json(SPLITS + "feature_columns.json")
LABEL_MAP = E.load_label_map(read_json(SPLITS + "label_map.json"))   # class index -> activity_id
K = len(LABEL_MAP)
ACTIVITY_IDS = list(LABEL_MAP.values())
print(K, "classes;", len(FEATURES), "features")


In [ ]:
import sagemaker
from sagemaker.processing import Processor, ProcessingInput, ProcessingOutput
sm_session = sagemaker.Session(boto_session=session)
ANALYZER = sagemaker.image_uris.retrieve("model-monitor", cfg["region"])

def run_analyzer(name, env, inputs, output_uri):
    """One-off Model Monitor analyzer job (ml.m5.large x1). inputs: [(s3_uri, container_dest, input_name)]."""
    proc = Processor(role=cfg["role_arn"], image_uri=ANALYZER, instance_count=1, instance_type="ml.m5.large",
                     volume_size_in_gb=20, max_runtime_in_seconds=3600, env=env, sagemaker_session=sm_session)
    proc.run(inputs=[ProcessingInput(source=s, destination=d, input_name=n) for s, d, n in inputs],
             outputs=[ProcessingOutput(source="/opt/ml/processing/output", destination=output_uri, output_name="result")],
             job_name=name, wait=True, logs=False)
    return output_uri
from sagemaker.model_monitor import DefaultModelMonitor
from sagemaker.model_monitor.dataset_format import DatasetFormat


## Monitor datasets (header CSV, feature_columns.json order, no label)

In [ ]:
train_uri = put_text(MON("data-quality/input/train.csv"), M.to_monitor_csv(read_parquet("train.parquet"), FEATURES))
prod_df = read_parquet("production.parquet")
prod_uri = put_text(MON("data-quality/input/production.csv"), M.to_monitor_csv(prod_df, FEATURES))
print(train_uri, prod_uri, len(prod_df), "production rows")


## Baseline on train (billable: one ml.m5.large processing job)

In [ ]:
BASELINE = MON("data-quality/baseline/")
monitor = DefaultModelMonitor(role=cfg["role_arn"], instance_count=1, instance_type="ml.m5.large",
                              volume_size_in_gb=20, max_runtime_in_seconds=3600, sagemaker_session=sm_session)
monitor.suggest_baseline(baseline_dataset=train_uri, dataset_format=DatasetFormat.csv(header=True),
                         output_s3_uri=BASELINE, wait=True, logs=False)
stats = read_json(BASELINE + "statistics.json")
cons = read_json(BASELINE + "constraints.json")
print(len(stats["features"]), "features profiled;", len(cons["features"]), "constraints")
cons_by_name = {c["name"]: c for c in cons["features"]}
display(pd.DataFrame([{"feature": f["name"], "type": f["inferred_type"],
                       "completeness_threshold": cons_by_name.get(f["name"], {}).get("completeness", {}).get("threshold")}
                      for f in stats["features"][:10]]))


## One-off monitoring job on production features (billable: one ml.m5.large processing job)

In [ ]:
stamp = time.strftime("%Y%m%d-%H%M%S")
RUN = MON(f"data-quality/runs/{stamp}/")
env = {"dataset_format": json.dumps(DatasetFormat.csv(header=True)), "dataset_source": "/opt/ml/processing/input/data",
       "output_path": "/opt/ml/processing/output", "publish_cloudwatch_metrics": "Disabled",
       "baseline_constraints": "/opt/ml/processing/baseline/constraints/constraints.json",
       "baseline_statistics": "/opt/ml/processing/baseline/stats/statistics.json"}
job_name = f"{cfg['prefix']}-data-monitor-{stamp}"
try:
    run_analyzer(job_name, env, [
        (prod_uri, "/opt/ml/processing/input/data", "data"),
        (BASELINE + "constraints.json", "/opt/ml/processing/baseline/constraints", "constraints"),
        (BASELINE + "statistics.json", "/opt/ml/processing/baseline/stats", "statistics")], RUN)
except Exception:
    report_failure(job_name)
    safe_publish({"DataJobFailed": 1})
    raise
safe_publish({"DataJobFailed": 0})


## Statistics, constraints and violations

In [ ]:
assert exists(RUN + "statistics.json"), "no statistics.json in run output - job incomplete; absent violations would not mean zero"
print("Run statistics:", RUN + "statistics.json")
print("Run constraints:", RUN + "constraints.json", "| exists:", exists(RUN + "constraints.json"))
v = M.parse_violations(read_bytes_or_none(RUN + "constraint_violations.json"))   # missing file = 0 violations
print("Violations:", v["count"], "->", v["status"])
display(pd.DataFrame(v["violations"], columns=["feature", "check_type", "description"]))
write_json(RUN + "violations_summary.json", v)
publish({"DataViolationCount": v["count"]})


## Cleanup

In [ ]:
# Cleanup: nothing here creates schedules or endpoints; list them, delete only temporary pamap2-har-mon* models.
for page in sm.get_paginator("list_monitoring_schedules").paginate(NameContains=cfg["prefix"]):
    for x in page["MonitoringScheduleSummaries"]:
        print("WARNING: monitoring schedule exists (not deleted):", x["MonitoringScheduleName"])
for page in sm.get_paginator("list_models").paginate(NameContains=f"{cfg['prefix']}-mon"):
    for m in page["Models"]:
        if m["ModelName"].startswith(f"{cfg['prefix']}-mon"):
            print("Deleting temporary model", m["ModelName"])
            sm.delete_model(ModelName=m["ModelName"])
for kind, pager, key, field in (("processing", "list_processing_jobs", "ProcessingJobSummaries", "ProcessingJobName"),
                                ("transform", "list_transform_jobs", "TransformJobSummaries", "TransformJobName")):
    for page in sm.get_paginator(pager).paginate(StatusEquals="InProgress", NameContains=cfg["prefix"]):
        for x in page[key]:
            print("WARNING: still running", kind, "job:", x[field])
print("Endpoints:", [e["EndpointName"] for p in sm.get_paginator("list_endpoints").paginate() for e in p["Endpoints"]])
